# Multi-Omics Feature-Selection Benchmark (Colab)

Notebook tự chứa: mỗi cell `%%writefile` chép **nguyên văn** một module preprocessing ra đĩa, rồi cell cuối chạy `main()` để benchmark **6 method** (FISHER, L1, MRMR, RELIEFF, SNR, ELMO) trên **cùng 1 split** và **cùng 1 mạng MLP**.

**Cách dùng:** chạy lần lượt từ trên xuống. Nhớ chỉnh `DATA_DIR` ở cell CONFIG cho khớp dữ liệu của bạn.

> Lưu ý: file omics ở dạng *(feature x sample)* nên `load_omics` đã `.T` để thành *(sample x feature)*. ELMO là method chậm nhất (RFE→IFS→SBS).

## 1. Cài thư viện & (tuỳ chọn) mount Google Drive

In [1]:
!pip -q install imbalanced-learn

# Mount Drive nếu dữ liệu nằm trên Google Drive (bỏ comment nếu cần):
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
%cd '/content/drive/MyDrive/HUST/Machine Learning'

/content/drive/MyDrive/HUST/Machine Learning


## 2. Ghi nguyên văn các module preprocessing ra đĩa
Mỗi cell dưới đây tạo lại 1 file `.py` y hệt bản gốc (không sửa code).

In [3]:
%%writefile MLP.py
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

try:
    from imblearn.over_sampling import SMOTE
    _HAS_SMOTE = True
except ImportError:
    _HAS_SMOTE = False


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class OmicsClassifierMLP(nn.Module):
    """
    MLP cho multi-omics.
    - input_norm (BatchNorm1d): chuẩn hoá per-feature NGAY TRONG mạng
      => bền vững với mọi kiểu preprocessing (không phụ thuộc bạn scale thế nào).
    - input_noise + input_dropout: regularize mạnh ở tầng vào, rất hợp với
      dữ liệu high-dim / low-sample điển hình của omics.
    - kiến trúc thu hẹp dần (taper), nhẹ hơn bản gốc để giảm overfit.
    """
    def __init__(self, input_dim, num_classes,
                 hidden_dims=(128, 64),
                 dropout=0.30, input_dropout=0.10, input_noise=0.05):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(input_dim)
        self.input_drop = nn.Dropout(input_dropout)
        self.input_noise = input_noise

        layers, prev = [], input_dim
        for i, h in enumerate(hidden_dims):
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout * (0.7 ** i)),   # dropout giảm dần theo độ sâu
            ]
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, num_classes)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_norm(x)
        if self.training and self.input_noise > 0:
            x = x + torch.randn_like(x) * self.input_noise
        x = self.input_drop(x)
        x = self.backbone(x)
        return self.head(x)


# ======================================================
# LOSS
# ======================================================
class FocalLoss(nn.Module):
    """Focal loss + class weights + label smoothing (cấu hình được)."""
    def __init__(self, gamma=1.5, weight=None, label_smoothing=0.05, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction="none")
        pt = torch.exp(-ce).clamp(1e-6, 1.0)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


def build_optimizer(model, lr, weight_decay):
    """Loại bias + tham số norm khỏi weight decay (best practice cho AdamW)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or "bias" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return optim.AdamW(
        [{"params": decay, "weight_decay": weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr)


# ======================================================
# TRAIN
# ======================================================
def step5_train_mlp(X_train_opt, y_train, num_classes,
                    epochs=250, batch_size=64, lr=1e-3, patience=30,
                    hidden_dims=(128, 64), dropout=0.30,
                    gamma=1.5, weight_decay=1e-4,
                    use_smote=False, use_class_weights=True,
                    val_size=0.15, seed=42, verbose=True):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_opt = np.asarray(X_train_opt, dtype=np.float32)
    y_train = np.asarray(y_train)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_opt, y_train, test_size=val_size,
        stratify=y_train, random_state=seed)

    # --- Mất cân bằng: CHỌN 1 trong 2, KHÔNG chồng cả SMOTE lẫn class-weight ---
    class_weight = None
    if use_smote and _HAS_SMOTE:
        k = int(max(1, min(5, np.bincount(y_tr).min() - 1)))
        try:
            X_tr, y_tr = SMOTE(random_state=seed, k_neighbors=k).fit_resample(X_tr, y_tr)
            use_class_weights = False
        except Exception as e:
            if verbose:
                print(f"[SMOTE bỏ qua] {e}")
    if use_class_weights:
        classes = np.unique(y_tr)
        cw = compute_class_weight("balanced", classes=classes, y=y_tr)
        full = np.ones(num_classes, dtype=np.float32)
        full[classes] = cw
        class_weight = torch.tensor(full, device=device)

    train_ds = TensorDataset(torch.from_numpy(X_tr).float(),
                             torch.from_numpy(y_tr).long())
    drop_last = len(train_ds) > batch_size      # tránh batch lẻ size=1 làm hỏng BatchNorm
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, drop_last=drop_last)

    X_val_t = torch.from_numpy(np.asarray(X_val, np.float32)).to(device)

    model = OmicsClassifierMLP(X_tr.shape[1], num_classes,
                               hidden_dims=hidden_dims, dropout=dropout).to(device)
    criterion = FocalLoss(gamma=gamma, weight=class_weight)
    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=max(1, len(train_loader)), pct_start=0.1)

    best_f1, wait = -1.0, 0
    best_state = copy.deepcopy(model.state_dict())
    min_delta = 1e-4

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).argmax(1).cpu().numpy()
        val_f1 = f1_score(y_val, val_pred, average="macro")

        if val_f1 > best_f1 + min_delta:
            best_f1, wait = val_f1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"Early stop @ epoch {epoch} | best macro-F1 = {best_f1:.4f}")
                break

    model.load_state_dict(best_state)
    return model


# ======================================================
# EVALUATE
# ======================================================
def evaluate_mlp(model, X_test, y_test):
    device = next(model.parameters()).device
    X_test = torch.from_numpy(np.asarray(X_test, np.float32)).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(X_test), dim=1).cpu().numpy()
    y_pred = probs.argmax(1)
    y_true = np.asarray(y_test)
    n_classes = probs.shape[1]

    try:
        if n_classes == 2:
            auc = roc_auc_score(y_true, probs[:, 1])
        else:
            auc = roc_auc_score(y_true, probs, multi_class="ovr",
                                average="macro", labels=np.arange(n_classes))
    except Exception:
        auc = np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "auc_macro": auc,
    }


# ======================================================
# CROSS-VALIDATION  (để so sánh CÔNG BẰNG các kiểu preprocessing)
# ======================================================
def cross_validate_mlp(X, y, num_classes, n_splits=5, seed=42, **train_kwargs):
    """
    Trả về (summary, rows). summary[metric] = (mean, std) qua các fold.
    Dùng để chấm điểm mỗi pipeline preprocessing và so sánh mean ± std,
    đáng tin hơn nhiều so với 1 lần split đơn lẻ.
    """
    X = np.asarray(X, np.float32)
    y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows = []
    for tr, te in skf.split(X, y):
        model = step5_train_mlp(X[tr], y[tr], num_classes,
                                seed=seed, verbose=False, **train_kwargs)
        rows.append(evaluate_mlp(model, X[te], y[te]))
    summary = {k: (float(np.nanmean([r[k] for r in rows])),
                   float(np.nanstd([r[k] for r in rows]))) for k in rows[0]}
    return summary, rows


Overwriting MLP.py


In [4]:
%%writefile ELMO.py
"""
Multi-Omics Feature Selection & Classification Pipeline
========================================================
Input : 4 CSV/TSV files  (rows = features, columns = samples)
        mrna.csv | mirna.csv | methylation.csv | cnv.csv
        A labels file: labels.csv  (one column, one row per sample, same order)

Correct leakage-free architecture
-----------------------------------
For each of the 5 outer folds:
  ├── TRAIN FOLD only:
  │     Step 1 – L1 LR          → drop zero-weight features
  │     Step 2 – LR-RFE         → rank surviving features
  │     Step 3 – IFS with LR    → find optimal feature count  (inner CV on train)
  │     Step 4 – SBS with LR    → refine feature set          (inner CV on train)
  │     Step 5 – Fit final LR on train with selected features
  └── TEST FOLD only:
        Evaluate: accuracy, F1-macro, precision-macro, recall-macro

Report mean ± std of all metrics across the 5 folds.

NOTE: Scaling (StandardScaler) is also fit on train fold only and applied
      to test fold — no leakage at any stage.
"""
from sklearn.decomposition import PCA
from sklearn.multiclass import OneVsRestClassifier
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score   # dùng trong inner-CV của các step 1-4

# ── MLP dùng chung (giống các pipeline khác) ──────────────────────────────────
from MLP import step5_train_mlp, evaluate_mlp, set_seed

warnings.filterwarnings("ignore")

# Siêu tham số MLP DÙNG CHUNG (đồng nhất với benchmark để so sánh công bằng)
MLP_KWARGS = dict(epochs=100, batch_size=32, lr=1e-3, patience=15, verbose=False)

# 5 chỉ số trả về từ evaluate_mlp
METRICS = ["accuracy", "f1_macro", "precision_macro", "recall_macro", "auc_macro"]

# ─────────────────────────────────────────────────────────────────────────────
# 0.  CONFIGURATION  –  edit paths / hyper-parameters here
# ─────────────────────────────────────────────────────────────────────────────
DATA_FILES = {
    "mrna":        "/content/drive/MyDrive/HUST/Machine Learning/Multi-Omics/BRCA_mRNA_aligned.csv",
    "mirna":       "/content/drive/MyDrive/HUST/Machine Learning/Multi-Omics/BRCA_miRNA_aligned.csv",
    "methylation": "/content/drive/MyDrive/HUST/Machine Learning/Multi-Omics/BRCA_Methy_aligned.csv",
    "cnv":         "/content/drive/MyDrive/HUST/Machine Learning/Multi-Omics/BRCA_CNV_aligned.csv",
}
LABELS_FILE = "/content/drive/MyDrive/HUST/Machine Learning/Multi-Omics/BRCA_label_num.csv"   # single column, no header OR header "label"

SEP          = ","
RANDOM_STATE = 42
N_FOLDS      = 5             # outer CV folds
N_INNER      = 4             # inner CV folds used inside IFS / SBS

# L1 LR (Step 1)
L1_C        = 0.1           # smaller → sparser solution
L1_SOLVER   = "liblinear"   # OvR + liblinear: nhanh (saga ~70'/fold tren 34k feature)
L1_MAX_ITER = 5000

# General LR (Steps 2–5)
LR_C        = 1.0
LR_SOLVER   = "lbfgs"
LR_MAX_ITER = 5000

RFE_STEP    = 0.1            # bo ~10%/vong thay vi 1 feature/vong -> nhanh hon nhieu
IFS_MAX_K   = 150           # tran IFS: khong quet qua 150 feature (du cho bai 5 lop)

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def load_omics(path: str, sep: str = ",") -> pd.DataFrame:
    """Load feature x sample CSV; return sample x feature (transposed)."""
    df = pd.read_csv(path, sep=sep, index_col=0)
    return df.T


def load_labels(path: str, sep: str = ",") -> np.ndarray:
    df = pd.read_csv(path, sep=sep, header= 0)
    return df.iloc[:, 0].values


def make_lr(C=LR_C, penalty="l2", solver=LR_SOLVER, max_iter=LR_MAX_ITER):
    return LogisticRegression(
        C=C, penalty=penalty, solver=solver,
        max_iter=max_iter, random_state=RANDOM_STATE,
    )


def impute_mean(X_train: np.ndarray, X_test: np.ndarray) -> tuple:
    """Mean-impute NaNs using train statistics only."""
    col_means = np.nanmean(X_train, axis=0)
    col_means  = np.where(np.isnan(col_means), 0.0, col_means)
    for X in (X_train, X_test):
        nans = np.where(np.isnan(X))
        X[nans] = np.take(col_means, nans[1])
    return X_train, X_test


def scale(X_train: np.ndarray, X_test: np.ndarray) -> tuple:
    """Fit StandardScaler on train, apply to both splits."""
    sc = StandardScaler()
    return sc.fit_transform(X_train), sc.transform(X_test)


def inner_cv_accuracy(X: np.ndarray, y: np.ndarray) -> float:
    """Mean accuracy of LR using N_INNER-fold stratified CV."""
    n_splits = min(N_INNER, int(np.min(np.bincount(y.astype(int)))))
    n_splits = max(n_splits, 2)
    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True,
                           random_state=RANDOM_STATE)
    accs = []
    for tr, va in cv.split(X, y):
        X_tr, X_va = X[tr], X[va]
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_va = sc.transform(X_va)
        lr = make_lr()
        lr.fit(X_tr, y[tr])
        accs.append(accuracy_score(y[va], lr.predict(X_va)))
    return float(np.mean(accs))


# ─────────────────────────────────────────────────────────────────────────────
# Step 1 – L1 feature selection  (train only)
# ─────────────────────────────────────────────────────────────────────────────

def step1_l1_selection(X_train_s: np.ndarray,
                       y_train:   np.ndarray,
                       feature_names: list) -> tuple:
    """
    Fit L1 LR on scaled train data.
    Returns boolean mask and list of surviving feature names.
    """
    base = LogisticRegression(
        penalty="l1", C=L1_C, solver=L1_SOLVER,
        max_iter=L1_MAX_ITER, random_state=RANDOM_STATE,
    )
    if L1_SOLVER == "liblinear" and len(np.unique(y_train)) > 2:
        clf = OneVsRestClassifier(base); clf.fit(X_train_s, y_train)
        importance = np.vstack([np.abs(e.coef_).ravel() for e in clf.estimators_]).max(axis=0)
    else:
        base.fit(X_train_s, y_train)
        importance = np.abs(base.coef_).max(axis=0)
    mask       = importance > 0
    selected   = [fn for fn, m in zip(feature_names, mask) if m]
    return mask, selected


# ─────────────────────────────────────────────────────────────────────────────
# Step 2 – LR-based RFE ranking  (train only)
# ─────────────────────────────────────────────────────────────────────────────

def step2_rfe_order(X_train_s: np.ndarray,
                    y_train:   np.ndarray) -> np.ndarray:
    """
    Run RFE down to 1 feature on scaled train data.
    Returns argsort of RFE ranking (best column index first).
    """
    rfe = RFE(estimator=make_lr(), n_features_to_select=1, step=RFE_STEP)
    rfe.fit(X_train_s, y_train)
    return np.argsort(rfe.ranking_)


# ─────────────────────────────────────────────────────────────────────────────
# Step 3 – Incremental Feature Selection  (train only, inner CV)
# ─────────────────────────────────────────────────────────────────────────────

def step3_ifs(X_train_ranked: np.ndarray,
              y_train:        np.ndarray,
              ranked_features: list) -> tuple:
    """
    Add one feature at a time in RFE rank order.
    Pick k that maximises inner-CV accuracy on the training fold.
    Returns (best_k, optimal_feature_names).
    """
    best_score, best_k = -1.0, 1
    k_max = min(X_train_ranked.shape[1], IFS_MAX_K)   # tran de IFS khong qua dat
    for k in range(1, k_max + 1):
        score = inner_cv_accuracy(X_train_ranked[:, :k], y_train)
        if score > best_score:
            best_score, best_k = score, k
    return best_k, ranked_features[:best_k]


# ─────────────────────────────────────────────────────────────────────────────
# Step 4 – Sequential Backward Selection  (train only, inner CV)
# ─────────────────────────────────────────────────────────────────────────────

def step4_sbs(X_train_opt: np.ndarray,
              y_train:     np.ndarray,
              optimal_features: list) -> tuple:
    """
    Greedily remove features when removal does not decrease inner-CV accuracy.
    Returns (list of kept column indices, final_feature_names).
    """
    current_cols     = list(range(X_train_opt.shape[1]))
    current_features = list(optimal_features)
    baseline         = inner_cv_accuracy(X_train_opt[:, current_cols], y_train)

    improved = True
    while improved and len(current_cols) > 1:
        improved   = False
        best_score = baseline
        best_drop  = -1

        for i in range(len(current_cols)):
            trial = current_cols[:i] + current_cols[i+1:]
            score = inner_cv_accuracy(X_train_opt[:, trial], y_train)
            if score >= best_score:
                best_score = score
                best_drop  = i

        if best_drop >= 0:
            current_cols.pop(best_drop)
            current_features.pop(best_drop)
            baseline = best_score
            improved = True

    return current_cols, current_features


# ─────────────────────────────────────────────────────────────────────────────
# MAIN  –  leakage-free outer loop
# ─────────────────────────────────────────────────────────────────────────────

def main():
    set_seed(RANDOM_STATE)
    # ── Load & merge data ──────────────────────────────────────────────────
    print("Loading omics data ...")
    dfs = {}
    for name, path in DATA_FILES.items():
        if not Path(path).exists():
            print(f"  [WARN] '{path}' not found – skipping '{name}'")
            continue
        dfs[name] = load_omics(path, SEP)
        print(f"  {name:<15}: {dfs[name].shape[0]} samples, "
              f"{dfs[name].shape[1]} features")

    if not dfs:
        print("\n[ERROR] No data files found. Set correct paths in CONFIG.")
        sys.exit(1)

    merged = pd.concat(list(dfs.values()), axis=1, join="inner")
    merged.columns = [
        f"{omic}__{feat}"
        for omic, df in dfs.items()
        for feat in df.columns
    ]
    print(f"\nMerged matrix : {merged.shape[0]} samples x "
          f"{merged.shape[1]} features")

    if not Path(LABELS_FILE).exists():
        print(f"\n[ERROR] Labels file '{LABELS_FILE}' not found.")
        sys.exit(1)

    y             = load_labels(LABELS_FILE, SEP).astype(int)
    feature_names = list(merged.columns)
    X             = merged.values.astype(float)
    num_classes   = len(np.unique(y))

    if len(y) != X.shape[0]:
        print(f"[ERROR] Label count ({len(y)}) != sample count ({X.shape[0]}).")
        sys.exit(1)

    # ── Outer 5-fold CV ────────────────────────────────────────────────────
    outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)

    fold_metrics  = []
    fold_features = []

    print("\n" + "="*65)
    print("OUTER 5-FOLD CROSS-VALIDATION")
    print("Train/test split is performed FIRST; all feature selection")
    print("steps (1-4) run exclusively on the training fold.")
    print("="*65)

    for fold_idx, (train_idx, test_idx) in enumerate(
            outer_cv.split(X, y), start=1):

        print(f"\n{'─'*65}")
        print(f"  FOLD {fold_idx}/{N_FOLDS}  "
              f"(train={len(train_idx)}, test={len(test_idx)})")
        print(f"{'─'*65}")

        # ── 0. Split ───────────────────────────────────────────────────────
        X_train_raw = X[train_idx].copy()
        X_test_raw  = X[test_idx].copy()
        y_train     = y[train_idx]
        y_test      = y[test_idx]

        # ── 0a. Impute NaNs (train statistics only) ────────────────────────
        X_train_raw, X_test_raw = impute_mean(X_train_raw, X_test_raw)

        # ── 0b. Scale (train statistics only) ─────────────────────────────
        X_train_s, X_test_s = scale(X_train_raw, X_test_raw)

        # ── STEP 1 – L1 selection (train only) ────────────────────────────
        mask, sel_features = step1_l1_selection(
            X_train_s, y_train, feature_names)

        X_train_sel = X_train_s[:, mask]
        X_test_sel  = X_test_s[:, mask]
        print(f"  Step 1 | L1 selection   : "
              f"{len(feature_names)} -> {len(sel_features)} features")

        if len(sel_features) == 0:
            print("  [WARN] No features survived L1 – skipping fold.")
            continue

        # ── STEP 2 – RFE ranking (train only) ─────────────────────────────
        rfe_order       = step2_rfe_order(X_train_sel, y_train)
        ranked_features = [sel_features[i] for i in rfe_order]
        X_train_ranked  = X_train_sel[:, rfe_order]
        X_test_ranked   = X_test_sel[:,  rfe_order]
        print(f"  Step 2 | RFE ranking    : top-5 = {ranked_features[:5]}")

        # ── STEP 3 – IFS (train only, inner CV) ───────────────────────────
        best_k, ifs_features = step3_ifs(
            X_train_ranked, y_train, ranked_features)

        X_train_ifs = X_train_ranked[:, :best_k]
        X_test_ifs  = X_test_ranked[:,  :best_k]
        print(f"  Step 3 | IFS optimal    : k={best_k}")

        # ── STEP 4 – SBS (train only, inner CV) ───────────────────────────
        kept_cols, final_features = step4_sbs(
            X_train_ifs, y_train, ifs_features)

        X_train_final = X_train_ifs[:, kept_cols]
        X_test_final  = X_test_ifs[:,  kept_cols]
        print(f"  Step 4 | SBS refined    : n={len(final_features)}  "
              f"-> {final_features}")

        # ── STEP 5 – Train MLP on TRAIN, evaluate on TEST (5 chỉ số) ────────
        mlp = step5_train_mlp(
            X_train_opt=X_train_final, y_train=y_train,
            num_classes=num_classes, **MLP_KWARGS,
        )
        eval_m = evaluate_mlp(mlp, X_test_final, y_test)   # 5 chỉ số (gồm AUC)

        m = {
            "fold":       fold_idx,
            "n_features": len(final_features),
            **{k: eval_m[k] for k in METRICS},
        }
        fold_metrics.append(m)
        fold_features.append(final_features)

        print(f"\n  Step 5 | Fold {fold_idx} test metrics (MLP):")
        for key in METRICS:
            print(f"           {key:<22}: {m[key]:.4f}")

    # ── Aggregate ──────────────────────────────────────────────────────────
    if not fold_metrics:
        print("\n[ERROR] No folds produced results.")
        sys.exit(1)

    df_folds = pd.DataFrame(fold_metrics)

    print("\n" + "="*65)
    print("FINAL SUMMARY  (mean +/- std across 5 folds)")
    print("="*65)
    print(f"  {'Metric':<24} {'Mean':>8}  {'Std':>8}")
    print("  " + "-"*44)

    summary_rows = []
    for col in METRICS:
        m = df_folds[col].mean()
        s = df_folds[col].std()
        print(f"  {col:<24} {m:>8.4f}  {s:>8.4f}")
        summary_rows.append({"Metric": col,
                              "Mean":   round(m, 4),
                              "Std":    round(s, 4)})

    print(f"\n  Mean features selected : {df_folds['n_features'].mean():.1f}")

    # ── Save ───────────────────────────────────────────────────────────────
    pd.DataFrame(summary_rows).to_csv("pipeline_metrics.csv", index=False)
    df_folds.to_csv("pipeline_fold_details.csv", index=False)
    pd.DataFrame([
        {"fold": i + 1, "feature": f}
        for i, feats in enumerate(fold_features)
        for f in feats
    ]).to_csv("fold_features.csv", index=False)

    print("\n" + "="*65)
    print("Output files:")
    print("  pipeline_metrics.csv      – mean +/- std summary")
    print("  pipeline_fold_details.csv – per-fold metrics & feature counts")
    print("  fold_features.csv         – features selected per fold")
    print("="*65)


if __name__ == "__main__":
    main()


Overwriting ELMO.py


In [5]:
%%writefile fisher.py
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif


DEFAULT_OMICS_TYPES = ("CNV", "Methy", "miRNA", "mRNA")


def as_dataframe(data: pd.DataFrame | np.ndarray | Sequence[Sequence[float]], feature_names: Sequence[str] | None = None) -> pd.DataFrame:
    if isinstance(data, pd.DataFrame):
        return data.copy()

    array = np.asarray(data)
    if feature_names is None:
        feature_names = [f"feature_{index}" for index in range(array.shape[1])]
    return pd.DataFrame(array, columns=list(feature_names))


def as_series(data: pd.Series | np.ndarray | Sequence[int], name: str) -> pd.Series:
    if isinstance(data, pd.Series):
        series = data.copy()
    else:
        series = pd.Series(np.asarray(data))
    series = series.reset_index(drop=True)
    series.name = name
    return series


def impute_from_train(X_train: pd.DataFrame, X_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_means = X_train.mean(numeric_only=True).fillna(0.0)
    return X_train.fillna(train_means), X_test.fillna(train_means)


def feature_metadata(feature_names: Sequence[str], scores: Sequence[float] | None = None) -> pd.DataFrame:
    rows = []
    for index, feature_name in enumerate(feature_names):
        gene_name = feature_name
        omics_type = "Unknown"
        row = {
            "feature": feature_name,
            "gene_name": gene_name,
            "omics_type": omics_type,
        }
        if scores is not None:
            row["score"] = float(scores[index])
        rows.append(row)
    return pd.DataFrame(rows)


def save_metadata(feature_names: Sequence[str], output_csv: str | Path, scores: Sequence[float] | None = None) -> pd.DataFrame:
    metadata = feature_metadata(feature_names, scores=scores)
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    metadata.to_csv(output_path, index=False)
    return metadata


def fisher_scores(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores, _ = f_classif(X, y)
    return np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)



def extract_features(
    X_train: pd.DataFrame | np.ndarray | Sequence[Sequence[float]],
    y_train: pd.Series | np.ndarray | Sequence[int],
    X_test: pd.DataFrame | np.ndarray | Sequence[Sequence[float]],
    y_test: pd.Series | np.ndarray | Sequence[int],
    *,
    k: int = 500,
    output_csv: str | Path = "fisher_selected_features.csv",
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    X_train_df = as_dataframe(X_train)
    X_test_df = as_dataframe(X_test, feature_names=X_train_df.columns)
    y_train_series = as_series(y_train, name="y_train")
    y_test_series = as_series(y_test, name="y_test")

    X_train_df, X_test_df = impute_from_train(X_train_df, X_test_df)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_df)
    X_test_scaled = scaler.transform(X_test_df)

    scores = fisher_scores(X_train_scaled, y_train_series.to_numpy())
    selected_count = min(int(k), X_train_df.shape[1])
    selected_indices = np.argsort(scores)[-selected_count:]
    selected_indices = np.sort(selected_indices)
    selected_columns = X_train_df.columns[selected_indices].tolist()

    X_train_selected = pd.DataFrame(X_train_scaled[:, selected_indices], columns=selected_columns, index=X_train_df.index)
    X_test_selected = pd.DataFrame(X_test_scaled[:, selected_indices], columns=selected_columns, index=X_test_df.index)

    save_metadata(selected_columns, output_csv, scores[selected_indices])
    return X_train_selected, y_train_series, X_test_selected, y_test_series


Overwriting fisher.py


In [6]:
%%writefile l1.py
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler


DEFAULT_OMICS_TYPES = ("CNV", "Methy", "miRNA", "mRNA")


def as_dataframe(data: pd.DataFrame | np.ndarray | Sequence[Sequence[float]], feature_names: Sequence[str] | None = None) -> pd.DataFrame:
    if isinstance(data, pd.DataFrame):
        return data.copy()

    array = np.asarray(data)
    if feature_names is None:
        feature_names = [f"feature_{index}" for index in range(array.shape[1])]
    return pd.DataFrame(array, columns=list(feature_names))


def as_series(data: pd.Series | np.ndarray | Sequence[int], name: str) -> pd.Series:
    if isinstance(data, pd.Series):
        series = data.copy()
    else:
        series = pd.Series(np.asarray(data))
    series = series.reset_index(drop=True)
    series.name = name
    return series


def impute_from_train(X_train: pd.DataFrame, X_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_means = X_train.mean(numeric_only=True).fillna(0.0)
    return X_train.fillna(train_means), X_test.fillna(train_means)


def infer_omics_type(feature_name: str) -> str:
    if "__" in feature_name:
        prefix = feature_name.split("__", 1)[0].strip().lower()
        mapping = {
            "cnv": "CNV",
            "methylation": "Methy",
            "methy": "Methy",
            "mirna": "miRNA",
            "mrna": "mRNA",
        }
        if prefix in mapping:
            return mapping[prefix]

    for omics_type in DEFAULT_OMICS_TYPES:
        if feature_name.endswith(f"_{omics_type}") or feature_name.endswith(f"__{omics_type}"):
            return omics_type

    return "Unknown"


def split_gene_and_omics(feature_name: str) -> tuple[str, str]:
    omics_type = infer_omics_type(feature_name)

    if "__" in feature_name:
        prefix, suffix = feature_name.split("__", 1)
        if prefix.strip().lower() in {"cnv", "methylation", "methy", "mirna", "mrna"}:
            return suffix, omics_type

    for suffix in (f"_{omics_type}", f"__{omics_type}"):
        if feature_name.endswith(suffix):
            return feature_name[: -len(suffix)], omics_type

    return feature_name, omics_type


def feature_metadata(feature_names: Sequence[str], scores: Sequence[float] | None = None) -> pd.DataFrame:
    rows = []
    for index, feature_name in enumerate(feature_names):
        gene_name, omics_type = split_gene_and_omics(feature_name)
        row = {
            "feature": feature_name,
            "gene_name": gene_name,
            "omics_type": omics_type,
        }
        if scores is not None:
            row["score"] = float(scores[index])
        rows.append(row)
    return pd.DataFrame(rows)


def save_metadata(feature_names: Sequence[str], output_csv: str | Path, scores: Sequence[float] | None = None) -> pd.DataFrame:
    metadata = feature_metadata(feature_names, scores=scores)
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    metadata.to_csv(output_path, index=False)
    return metadata


def make_lr_l1(C: float, solver: str, max_iter: int, random_state: int) -> LogisticRegression:
    return LogisticRegression(
        C=C,
        penalty="l1",
        solver=solver,
        max_iter=max_iter,
        random_state=random_state,
    )


def l1_scores(X: np.ndarray, y: np.ndarray, *, C: float, solver: str, max_iter: int, random_state: int) -> np.ndarray:
    base = LogisticRegression(
        C=C, penalty="l1", solver=solver,
        max_iter=max_iter, random_state=random_state,
    )
    n_classes = len(np.unique(y))
    if solver == "liblinear" and n_classes > 2:
        # liblinear khong lam multinomial -> One-vs-Rest (nhanh, dung y do goc multi_class='auto')
        clf = OneVsRestClassifier(base)
        clf.fit(X, y)
        return np.vstack([np.abs(e.coef_).ravel() for e in clf.estimators_]).max(axis=0)
    base.fit(X, y)
    coefficients = np.abs(base.coef_)
    return coefficients if coefficients.ndim == 1 else coefficients.max(axis=0)


def extract_features(
    X_train: pd.DataFrame | np.ndarray | Sequence[Sequence[float]],
    y_train: pd.Series | np.ndarray | Sequence[int],
    X_test: pd.DataFrame | np.ndarray | Sequence[Sequence[float]],
    y_test: pd.Series | np.ndarray | Sequence[int],
    *,
    k: int = 500,
    output_csv: str | Path = "l1_selected_features.csv",
    random_state: int = 42,
    C: float = 0.1,
    solver: str = "liblinear",  # liblinear + OvR: nhanh; saga ~70'/fold tren 34k feature
    max_iter: int = 5000,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    X_train_df = as_dataframe(X_train)
    X_test_df = as_dataframe(X_test, feature_names=X_train_df.columns)
    y_train_series = as_series(y_train, name="y_train")
    y_test_series = as_series(y_test, name="y_test")

    X_train_df, X_test_df = impute_from_train(X_train_df, X_test_df)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_df)
    X_test_scaled = scaler.transform(X_test_df)

    scores = l1_scores(
        X_train_scaled,
        y_train_series.to_numpy(),
        C=C,
        solver=solver,
        max_iter=max_iter,
        random_state=random_state,
    )

    selected_count = min(int(k), X_train_df.shape[1])
    selected_indices = np.argsort(scores)[-selected_count:]
    selected_indices = np.sort(selected_indices)
    selected_columns = X_train_df.columns[selected_indices].tolist()

    X_train_selected = pd.DataFrame(X_train_scaled[:, selected_indices], columns=selected_columns, index=X_train_df.index)
    X_test_selected = pd.DataFrame(X_test_scaled[:, selected_indices], columns=selected_columns, index=X_test_df.index)

    save_metadata(selected_columns, output_csv, scores[selected_indices])
    return X_train_selected, y_train_series, X_test_selected, y_test_series


Overwriting l1.py


In [7]:
%%writefile snr.py
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd


DEFAULT_OMICS_TYPES = ("CNV", "Methy", "miRNA", "mRNA")


def as_dataframe(data: pd.DataFrame | np.ndarray | Sequence[Sequence[float]], feature_names: Sequence[str] | None = None) -> pd.DataFrame:
    if isinstance(data, pd.DataFrame):
        return data.copy()

    array = np.asarray(data)
    if feature_names is None:
        feature_names = [f"feature_{index}" for index in range(array.shape[1])]
    return pd.DataFrame(array, columns=list(feature_names))


def as_series(data: pd.Series | np.ndarray | Sequence[int], name: str) -> pd.Series:
    if isinstance(data, pd.Series):
        series = data.copy()
    else:
        series = pd.Series(np.asarray(data))
    series = series.reset_index(drop=True)
    series.name = name
    return series


def impute_from_train(X_train: pd.DataFrame, X_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_means = X_train.mean(numeric_only=True).fillna(0.0)
    return X_train.fillna(train_means), X_test.fillna(train_means)


def infer_omics_type(feature_name: str) -> str:
    if "__" in feature_name:
        prefix = feature_name.split("__", 1)[0].strip().lower()
        mapping = {
            "cnv": "CNV",
            "methylation": "Methy",
            "methy": "Methy",
            "mirna": "miRNA",
            "mrna": "mRNA",
        }
        if prefix in mapping:
            return mapping[prefix]

    for omics_type in DEFAULT_OMICS_TYPES:
        if feature_name.endswith(f"_{omics_type}") or feature_name.endswith(f"__{omics_type}"):
            return omics_type

    return "Unknown"


def split_gene_and_omics(feature_name: str) -> tuple[str, str]:
    omics_type = infer_omics_type(feature_name)

    if "__" in feature_name:
        prefix, suffix = feature_name.split("__", 1)
        if prefix.strip().lower() in {"cnv", "methylation", "methy", "mirna", "mrna"}:
            return suffix, omics_type

    for suffix in (f"_{omics_type}", f"__{omics_type}"):
        if feature_name.endswith(suffix):
            return feature_name[: -len(suffix)], omics_type

    return feature_name, omics_type


def feature_metadata(feature_names: Sequence[str], scores: Sequence[float] | None = None) -> pd.DataFrame:
    rows = []
    for index, feature_name in enumerate(feature_names):
        gene_name, omics_type = split_gene_and_omics(feature_name)
        row = {
            "feature": feature_name,
            "gene_name": gene_name,
            "omics_type": omics_type,
        }
        if scores is not None:
            row["score"] = float(scores[index])
        rows.append(row)
    return pd.DataFrame(rows)


def save_metadata(feature_names: Sequence[str], output_csv: str | Path, scores: Sequence[float] | None = None) -> pd.DataFrame:
    metadata = feature_metadata(feature_names, scores=scores)
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    metadata.to_csv(output_path, index=False)
    return metadata


def snr_scores(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    classes = np.unique(y)
    scores = np.zeros(X.shape[1], dtype=float)
    eps = 1e-12

    for class_label in classes:
        class_mask = y == class_label
        rest_mask = ~class_mask
        class_mean = np.nanmean(X[class_mask], axis=0)
        rest_mean = np.nanmean(X[rest_mask], axis=0)
        class_std = np.nanstd(X[class_mask], axis=0)
        rest_std = np.nanstd(X[rest_mask], axis=0)
        scores = np.maximum(scores, np.abs(class_mean - rest_mean) / (class_std + rest_std + eps))

    return np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

def extract_features(
    X_train: pd.DataFrame | np.ndarray | Sequence[Sequence[float]],
    y_train: pd.Series | np.ndarray | Sequence[int],
    X_test: pd.DataFrame | np.ndarray | Sequence[Sequence[float]],
    y_test: pd.Series | np.ndarray | Sequence[int],
    *,
    k_per_class: int = 10,
    k_total: int | None = None,
    output_csv: str | Path = "snr_selected_features.csv",
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """SNR-based feature selection with one-vs-all ranking for multiclass data."""

    X_train_df = as_dataframe(X_train)
    X_test_df = as_dataframe(X_test, feature_names=X_train_df.columns)
    y_train_series = as_series(y_train, name="y_train")
    y_test_series = as_series(y_test, name="y_test")

    X_train_df, X_test_df = impute_from_train(X_train_df, X_test_df)

    X_train_array = X_train_df.to_numpy(dtype=float, copy=True)
    X_test_array = X_test_df.to_numpy(dtype=float, copy=True)
    y_train_array = y_train_series.to_numpy()

    # --- Global top-k mode: rank ALL features by multiclass SNR score ----
    if k_total is not None:
        scores_all = snr_scores(X_train_array, y_train_array)
        k_actual = min(int(k_total), X_train_array.shape[1])
        sel = np.sort(np.argsort(scores_all)[::-1][:k_actual])
        cols = X_train_df.columns[sel].tolist()
        X_train_selected = pd.DataFrame(X_train_array[:, sel], columns=cols, index=X_train_df.index)
        X_test_selected = pd.DataFrame(X_test_array[:, sel], columns=cols, index=X_test_df.index)
        save_metadata(cols, output_csv, scores_all[sel])
        return X_train_selected, y_train_series, X_test_selected, y_test_series

    classes = np.unique(y_train_array)
    selected_features = set()
    selected_scores: dict[int, float] = {}

    for class_label in classes:
        y_binary = (y_train_array == class_label).astype(int)
        scores = snr_scores(X_train_array, y_binary)

        # Chỉ giữ top-k feature CÓ SNR CAO NHẤT cho mỗi lớp.
        # (Bản cũ còn lấy thêm bottom-k qua `neg_idx` -> chọn nhầm các feature
        #  ÍT phân biệt nhất; đã loại bỏ.)
        pos_idx = np.argsort(scores)[::-1][:k_per_class]

        for index in pos_idx:
            selected_features.add(int(index))
            selected_scores[int(index)] = float(scores[int(index)])

    selected_idx = np.array(sorted(selected_features))
    selected_columns = X_train_df.columns[selected_idx].tolist()

    X_train_selected = pd.DataFrame(X_train_array[:, selected_idx], columns=selected_columns, index=X_train_df.index)
    X_test_selected = pd.DataFrame(X_test_array[:, selected_idx], columns=selected_columns, index=X_test_df.index)

    ordered_scores = np.array([selected_scores[int(index)] for index in selected_idx], dtype=float)
    save_metadata(selected_columns, output_csv, ordered_scores)
    return X_train_selected, y_train_series, X_test_selected, y_test_series


Overwriting snr.py


In [8]:
%%writefile ReliefF.py
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import cdist

# Default hyper-parameters (override via function arguments)
_RELIEF_ITER    = 200
_RELIEF_K       = 10
_RANDOM_STATE   = 42


# ─────────────────────────────────────────────────────────────────────────────
# Core algorithm
# ─────────────────────────────────────────────────────────────────────────────

def _relieff_weights(
    X:      np.ndarray,
    y:      np.ndarray,
    n_iter: int = _RELIEF_ITER,
    k:      int = _RELIEF_K,
    seed:   int = _RANDOM_STATE,
) -> np.ndarray:
    """
    Compute multi-class ReliefF feature weights.

    At each iteration a random instance i is drawn; weights are updated by:
      • subtracting mean absolute difference to its k nearest hits  (same class)
      • adding  mean absolute difference to its k nearest misses (other classes),
        scaled by the class prior ratio  p(c) / (1 − p(y_i))

    Parameters
    ----------
    X      : (n_samples, n_features)  training data
    y      : (n_samples,)             integer class labels
    n_iter : number of random sampling iterations
    k      : number of nearest neighbours considered per iteration
    seed   : random seed for reproducibility

    Returns
    -------
    W : (n_features,)  relief weight per feature (higher = more discriminative)
    """
    rng       = np.random.default_rng(seed)
    n, f      = X.shape
    W         = np.zeros(f, dtype=np.float64)
    classes   = np.unique(y)
    class_p   = {c: (y == c).mean() for c in classes}
    class_idx = {c: np.where(y == c)[0] for c in classes}

    # Pre-compute full distance matrix; diagonal set to inf so self is never
    # selected as a neighbour
    dist_mat = cdist(X, X, metric='euclidean')
    np.fill_diagonal(dist_mat, np.inf)

    for _ in range(n_iter):
        i       = rng.integers(n)
        xi, yi  = X[i], y[i]

        # ── Nearest hits ────────────────────────────────────────────────────
        same = class_idx[yi]
        same = same[same != i]
        if len(same) > 0:
            d_same = dist_mat[i, same]
            hits   = same[np.argpartition(d_same, min(k, len(same) - 1))[:k]]
            W     -= np.abs(X[hits] - xi).mean(axis=0) / n_iter

        # ── Nearest misses (per opposing class) ─────────────────────────────
        for c in classes:
            if c == yi:
                continue
            other = class_idx[c]
            if len(other) > 0:
                d_oth  = dist_mat[i, other]
                misses = other[np.argpartition(d_oth, min(k, len(other) - 1))[:k]]
                weight = class_p[c] / (1.0 - class_p[yi] + 1e-10)
                W     += weight * np.abs(X[misses] - xi).mean(axis=0) / n_iter

    return W


# ─────────────────────────────────────────────────────────────────────────────
# Public interface
# ─────────────────────────────────────────────────────────────────────────────

def relieff_extract_features(
    X_train:       np.ndarray,
    y_train:       np.ndarray,
    X_test:        np.ndarray,
    y_test:        np.ndarray,
    feature_names: list[str],
    k:             int,
    out_dir:       str = ".",
    fold:          int = 1,
    n_iter:        int = _RELIEF_ITER,
    k_neighbors:   int = _RELIEF_K,
    seed:          int = _RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Apply ReliefF feature weighting and return reduced data arrays.

    Weights are computed using X_train / y_train ONLY; the same top-k
    column indices are then applied to X_test (no leakage).

    Parameters
    ----------
    X_train       : (n_train, n_features)  scaled training data
    y_train       : (n_train,)             training labels
    X_test        : (n_test,  n_features)  scaled test data
    y_test        : (n_test,)              test labels (passed through unchanged)
    feature_names : list of column names matching axis-1 of X_train
    k             : number of top features to keep
    out_dir       : directory to write the selected-feature CSV
    fold          : current fold number (used in the output filename)
    n_iter        : number of ReliefF sampling iterations
    k_neighbors   : number of nearest neighbours per iteration
    seed          : random seed for reproducibility

    Returns
    -------
    X_train_sel, y_train, X_test_sel, y_test
        Arrays with only the top-k columns; y arrays are unchanged.

    Side-effect
    -----------
    Writes  {out_dir}/relieff_selected_features_fold{fold}.csv
    with columns [feature, omics_modality, relieff_score].
    """
    k_actual = min(k, X_train.shape[1])

    # ── Compute weights on train only ────────────────────────────────────────
    weights = _relieff_weights(X_train, y_train, n_iter=n_iter, k=k_neighbors, seed=seed)
    top_idx = np.argsort(weights)[::-1][:k_actual]

    selected_names   = [feature_names[i] for i in top_idx]
    selected_weights = weights[top_idx]

    print(f"  ReliefF selection   : {X_train.shape[1]:,} → {k_actual} features")

    # ── Save selected feature list ───────────────────────────────────────────
    _save_selected_features(
        feature_names = selected_names,
        scores        = selected_weights,
        method        = "relieff",
        out_dir       = out_dir,
        fold          = fold,
    )

    return (
        X_train[:, top_idx],
        y_train,
        X_test[:, top_idx],
        y_test,
    )


# ─────────────────────────────────────────────────────────────────────────────
# I/O helper (shared pattern with fisher module)
# ─────────────────────────────────────────────────────────────────────────────

def _save_selected_features(
    feature_names: list[str],
    scores:        np.ndarray | None,
    method:        str,
    out_dir:       str,
    fold:          int,
) -> None:
    """
    Persist selected feature names to CSV, annotated with their omics modality.

    Feature names are expected to follow the convention  '<modality>__<gene>'
    (as produced by utils.load_omics).  The modality is extracted from the
    prefix; if no prefix is found the column is labelled 'unknown'.

    Output columns
    --------------
    feature        : full feature name (e.g. 'mRNA__TP53')
    omics_modality : prefix before '__'  (e.g. 'mRNA')
    {method}_score : numeric score if provided, else absent
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    records = []
    for i, name in enumerate(feature_names):
        parts    = name.split("__", 1)
        modality = parts[0] if len(parts) == 2 else "unknown"
        row: dict = {"feature": name, "omics_modality": modality}
        if scores is not None:
            row[f"{method}_score"] = float(scores[i])
        records.append(row)

    df   = pd.DataFrame(records)
    path = out / f"{method}_selected_features_fold{fold}.csv"
    df.to_csv(path, index=False)
    print(f"  Selected features saved → {path}")


Overwriting ReliefF.py


In [9]:
%%writefile mRMR.py
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif

# Default hyper-parameters (override via function arguments)
_MRMR_K       = 50
_RANDOM_STATE = 42


# -----------------------------------------------------------------------------
# Core algorithm  (relevance = MI; redundancy = correlation, INCREMENTAL)
# -----------------------------------------------------------------------------

def _mrmr_select(
    X:                 np.ndarray,
    y:                 np.ndarray,
    k_select:          int   = _MRMR_K,
    seed:              int   = _RANDOM_STATE,
    redundancy_weight: float = 1.0,
) -> np.ndarray:
    """
    Greedy mRMR selection with a TUNABLE redundancy penalty.

        score(f) = relevance(f) - redundancy_weight * mean|corr(f, selected)|

    Two changes vs the classic implementation, aimed at downstream MLPs:

      * Relevance (mutual information) is normalised to [0, 1] so that the
        redundancy term (mean |Pearson corr|, also in [0, 1]) is on the same
        scale; `redundancy_weight` (alpha) is therefore interpretable:
          - alpha = 1.0  : strong redundancy penalty (classic mRMR behaviour,
                            tends to push selection onto orthogonal CNV features)
          - alpha = 0.0  : pure MaxRel (top mutual information; mRNA-rich)
        Neural nets benefit from correlated/redundant features, so a LOWER
        alpha usually selects an MLP-friendlier set.

      * The redundancy term is accumulated INCREMENTALLY (correlation of each
        newly-selected feature vs all others) instead of materialising the full
        F x F correlation matrix -> memory O(F) rather than O(F^2) (safe at 34k
        features). The maths is identical to the full-matrix version.
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)

    # -- Relevance: multi-class mutual information, scaled to [0, 1] -----------
    mi = mutual_info_classif(X, y, random_state=seed)
    mi_norm = mi / (mi.max() + 1e-12)

    # -- Standardise columns (centre + unit L2) so dot product = Pearson corr --
    Xc   = X - X.mean(axis=0, keepdims=True)
    norm = np.sqrt((Xc ** 2).sum(axis=0, keepdims=True))
    norm[norm == 0] = 1.0                      # constant columns -> avoid /0
    Xs = Xc / norm                             # (n, F)

    F        = X.shape[1]
    k_actual = min(k_select, F)
    redund_sum    = np.zeros(F)
    selected_mask = np.zeros(F, dtype=bool)
    selected: list[int] = []

    for _ in range(k_actual):
        if not selected:
            best = int(np.argmax(mi_norm))
        else:
            score = mi_norm - redundancy_weight * (redund_sum / len(selected))
            score[selected_mask] = -np.inf
            best = int(np.argmax(score))
        selected.append(best)
        selected_mask[best] = True

        # corr(best, :) = Xs[:, best] . Xs   (length F); set corr(best,best)=0
        corr_best = np.abs(Xs[:, best] @ Xs)
        corr_best[best] = 0.0
        redund_sum += corr_best

    return np.array(selected)


# -----------------------------------------------------------------------------
# Public interface
# -----------------------------------------------------------------------------

def mrmr_extract_features(
    X_train:       np.ndarray,
    y_train:       np.ndarray,
    X_test:        np.ndarray,
    y_test:        np.ndarray,
    feature_names: list[str],
    k:             int = _MRMR_K,
    out_dir:       str = ".",
    fold:          int = 1,
    seed:          int = _RANDOM_STATE,
    redundancy_weight: float = 1.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Apply mRMR feature selection and return reduced data arrays.

    Selection is performed using X_train / y_train ONLY; the same top-k
    column indices are then applied to X_test (no leakage).

    Parameters
    ----------
    X_train       : (n_train, n_features)  scaled training data
    y_train       : (n_train,)             training labels
    X_test        : (n_test,  n_features)  scaled test data
    y_test        : (n_test,)              test labels (passed through unchanged)
    feature_names : list of column names matching axis-1 of X_train
    k             : number of top features to keep
    out_dir       : directory to write the selected-feature CSV
    fold          : current fold number (used in the output filename)
    seed          : random seed for mutual_info_classif

    Returns
    -------
    X_train_sel, y_train, X_test_sel, y_test
        Arrays with only the top-k columns; y arrays are unchanged.

    Side-effect
    -----------
    Writes  {out_dir}/mrmr_selected_features_fold{fold}.csv
    with columns [feature, omics_modality].
    """
    k_actual = min(k, X_train.shape[1])

    # ── Select on train only ─────────────────────────────────────────────────
    top_idx = _mrmr_select(
        X_train, y_train,
        k_select          = k_actual,
        seed              = seed,
        redundancy_weight = redundancy_weight,
    )

    selected_names = [feature_names[i] for i in top_idx]

    print(f"  mRMR selection      : {X_train.shape[1]:,} → {k_actual} features")

    # ── Save selected feature list ───────────────────────────────────────────
    _save_selected_features(
        feature_names = selected_names,
        scores        = None,           # mRMR has no single scalar score
        method        = "mrmr",
        out_dir       = out_dir,
        fold          = fold,
    )

    return (
        X_train[:, top_idx],
        y_train,
        X_test[:, top_idx],
        y_test,
    )


# ─────────────────────────────────────────────────────────────────────────────
# I/O helper (shared pattern with fisher / relieff modules)
# ─────────────────────────────────────────────────────────────────────────────

def _save_selected_features(
    feature_names: list[str],
    scores:        np.ndarray | None,
    method:        str,
    out_dir:       str,
    fold:          int,
) -> None:
    """
    Persist selected feature names to CSV, annotated with their omics modality.

    Feature names are expected to follow the convention  '<modality>__<gene>'
    (as produced by utils.load_omics).  The modality is extracted from the
    prefix; if no prefix is found the column is labelled 'unknown'.

    Output columns
    --------------
    feature        : full feature name (e.g. 'mRNA__TP53')
    omics_modality : prefix before '__'  (e.g. 'mRNA')
    {method}_score : numeric score if provided, else absent
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    records = []
    for i, name in enumerate(feature_names):
        parts    = name.split("__", 1)
        modality = parts[0] if len(parts) == 2 else "unknown"
        row: dict = {"feature": name, "omics_modality": modality}
        if scores is not None:
            row[f"{method}_score"] = float(scores[i])
        records.append(row)

    df   = pd.DataFrame(records)
    path = out / f"{method}_selected_features_fold{fold}.csv"
    df.to_csv(path, index=False)
    print(f"  Selected features saved → {path}")


Overwriting mRMR.py


## 3. CONFIG — chỉnh đường dẫn dữ liệu

In [10]:
# ============================================================================
# CONFIG cho Colab  —  CHỈNH đường dẫn dữ liệu ở đây
# ============================================================================
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 5 module preprocessing + ELMO (đã ghi ra đĩa ở các cell %%writefile phía trên)
import fisher
import l1
import snr
import ELMO
from ReliefF import relieff_extract_features
from mRMR import mrmr_extract_features
from MLP import step5_train_mlp, evaluate_mlp, set_seed

# >>> SỬA đường dẫn cho khớp dữ liệu của bạn trên Colab/Drive <<<
DATA_DIR = Path("/content/drive/MyDrive/HUST/Machine Learning/Multi-Omics")
DATA_FILES = {
    "CNV":   DATA_DIR / "BRCA_CNV_aligned.csv",
    "Methy": DATA_DIR / "BRCA_Methy_aligned.csv",
    "miRNA": DATA_DIR / "BRCA_miRNA_aligned.csv",
    "mRNA":  DATA_DIR / "BRCA_mRNA_aligned.csv",
}
SEP         = ","
LABELS_FILE = DATA_DIR / "BRCA_label_num.csv"   # 1 cột 'Label', không có index

OUT_DIR = "benchmark_out"
SEED    = 42

# Siêu tham số MLP DÙNG CHUNG cho mọi method (so sánh công bằng)
# Tinh chinh cho BRCA MLOmics (671 mau, 5 lop mat can bang, ~50 feature):
#   - capacity vua: (192,96) + dropout 0.35 (BN + input-noise + label-smooth giu khoi overfit)
#   - lr thap + nhieu epoch + patience rong: early-stop theo val macro-F1 tu tim diem tot nhat
#   - gamma focal 2.0 + weight_decay 3e-4: ep manh hon vao cac lop hiem (Normal-like, Her2...)
# >>> Do them: chinh hidden_dims/dropout/lr/gamma/weight_decay o day, CHON theo validation.
MLP_KWARGS = dict(epochs=200, batch_size=32, lr=6e-4, patience=30, verbose=False,
                  hidden_dims=(192, 96), dropout=0.35, gamma=2.0, weight_decay=3e-4)

# Số đặc trưng giữ lại cho từng method
# --- Ngan sach = 50 cho 4 method co dinh; SNR & ELMO GIU ban chat ---------
# 4 method top-k chon DUNG 50 dac trung: Fisher/L1/mRMR/ReliefF
# SNR  : DUNG global top-k = K_FIXED (50) de SO CONG BANG (da bo bottom-k loi)
# ELMO : GIU adaptive L1 -> RFE -> IFS -> SBS (tu chon so feature) -> ~50-70
# (ca hai method thich nghi giu nguyen ban chat; cot n_feat phan anh so thuc te)
K_FIXED        = 50           # sweep thu: 30 / 50 / 100 / 200
K_FISHER       = K_FIXED
K_L1           = K_FIXED
K_MRMR         = K_FIXED
K_RELIEFF      = K_FIXED
K_SNR_PERCLASS = 10           # (giu lai, KHONG dung khi da bat k_total ben duoi)
K_SNR_TOTAL    = K_FIXED      # SNR dung global top-k = 50 -> SO CONG BANG voi 4 ranker kia

# --- mRMR: trong so penalty redundancy (alpha) --------------------------
# Relevance da duoc chuan hoa ve [0,1]; score = relevance - alpha * redundancy.
#   alpha = 1.0  -> phat redundancy manh (kieu mRMR co dien, day sang CNV)
#   alpha = 0.0  -> chi MaxRel (top mutual information, giau mRNA)
# MLP huong loi tu feature tuong quan => alpha thap phu hop hon. Sweep: 0/0.3/0.5/1.
MRMR_REDUNDANCY_WEIGHT = 0.3

# --- ReliefF: so vong lap (uoc luong trong so chinh xac/on dinh hon) -----
RELIEFF_N_ITER = 500          # tang tu 200; >= so mau train cang on dinh

# Số fold cho Stratified K-Fold cross-validation
N_FOLDS = 5


## 4. Hàm load dữ liệu, adapter cho từng method, và benchmark

In [11]:
def load_omics(path: str, sep: str) -> pd.DataFrame:
    """Đọc 1 file omics -> DataFrame (hàng = sample, cột = feature).

    Giả định cột đầu là id sample (index). Nếu file của bạn là feature x sample
    thì bỏ comment dòng `.T` để chuyển vị.
    """
    df = pd.read_csv(path, sep=sep, index_col=0)
    df = df.T             # file ở dạng (feature x sample) -> chuyển thành (sample x feature)
    return df


def load_labels(path: str, sep: str) -> np.ndarray:
    """Đọc nhãn -> mảng 1 chiều theo đúng thứ tự sample.

    File BRCA_label_num.csv chỉ có 1 cột 'Label', KHÔNG có cột index.
    """
    df = pd.read_csv(path, sep=sep)
    return df.iloc[:, 0].to_numpy()


def load_dataset() -> tuple[np.ndarray, np.ndarray, list[str], int]:
    """Load + merge các omics, trả về (X, y, feature_names, num_classes)."""
    print("Loading omics data ...")
    dfs = {}
    for name, path in DATA_FILES.items():
        if not Path(path).exists():
            print(f"  [WARN] '{path}' not found - skipping '{name}'")
            continue
        dfs[name] = load_omics(path, SEP)
        print(f"  {name:<8}: {dfs[name].shape[0]} samples, {dfs[name].shape[1]} features")

    if not dfs:
        print("\n[ERROR] Không tìm thấy file dữ liệu nào. Hãy chỉnh CONFIG.DATA_FILES.")
        sys.exit(1)

    merged = pd.concat(list(dfs.values()), axis=1, join="inner")
    merged.columns = [f"{omic}__{feat}" for omic, df in dfs.items() for feat in df.columns]
    feature_names = list(merged.columns)
    print(f"\nMerged matrix : {merged.shape[0]} samples x {merged.shape[1]} features")

    if not Path(LABELS_FILE).exists():
        print(f"\n[ERROR] Labels file '{LABELS_FILE}' not found.")
        sys.exit(1)

    y_raw = load_labels(LABELS_FILE, SEP)
    X = merged.values.astype(float)
    if len(y_raw) != X.shape[0]:
        print(f"[ERROR] Label count ({len(y_raw)}) != sample count ({X.shape[0]}).")
        sys.exit(1)

    le = LabelEncoder()
    y = le.fit_transform(np.asarray(y_raw))
    return X, y, feature_names, len(le.classes_)


# ==============================================================================
# HELPERS preprocessing chung cho ReliefF / mRMR (cần dữ liệu đã scale)
# ==============================================================================
def _impute_scale(X_train: np.ndarray, X_test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Impute NaN bằng mean của TRAIN rồi StandardScaler (fit trên train, áp lên test)."""
    col_mean = np.nanmean(X_train, axis=0)
    col_mean = np.where(np.isnan(col_mean), 0.0, col_mean)
    X_train = np.where(np.isnan(X_train), col_mean, X_train)
    X_test = np.where(np.isnan(X_test), col_mean, X_test)
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test)


# ==============================================================================
# ADAPTERS: mỗi method -> (X_train_opt, y_train, X_test_opt, y_test) dạng numpy
#   Giữ nguyên hàm preprocessing gốc, chỉ chuẩn hoá input/output.
# ==============================================================================
def prep_fisher(Xtr, ytr, Xte, yte, feature_names, fold):
    Xtr_df = pd.DataFrame(Xtr, columns=feature_names)
    Xte_df = pd.DataFrame(Xte, columns=feature_names)
    a, b, c, d = fisher.extract_features(
        Xtr_df, ytr, Xte_df, yte,
        k=K_FISHER, output_csv=f"{OUT_DIR}/fisher_selected_features.csv",
    )
    return a.to_numpy(np.float32), b.to_numpy(), c.to_numpy(np.float32), d.to_numpy()


def prep_l1(Xtr, ytr, Xte, yte, feature_names, fold):
    Xtr_df = pd.DataFrame(Xtr, columns=feature_names)
    Xte_df = pd.DataFrame(Xte, columns=feature_names)
    a, b, c, d = l1.extract_features(
        Xtr_df, ytr, Xte_df, yte,
        k=K_L1, output_csv=f"{OUT_DIR}/l1_selected_features.csv", random_state=SEED,
    )
    return a.to_numpy(np.float32), b.to_numpy(), c.to_numpy(np.float32), d.to_numpy()


def prep_snr(Xtr, ytr, Xte, yte, feature_names, fold):
    Xtr_df = pd.DataFrame(Xtr, columns=feature_names)
    Xte_df = pd.DataFrame(Xte, columns=feature_names)
    a, b, c, d = snr.extract_features(
        Xtr_df, ytr, Xte_df, yte,
        k_total=K_SNR_TOTAL, output_csv=f"{OUT_DIR}/snr_selected_features.csv",
    )
    return a.to_numpy(np.float32), b.to_numpy(), c.to_numpy(np.float32), d.to_numpy()


def prep_relieff(Xtr, ytr, Xte, yte, feature_names, fold):
    # ReliefF dùng khoảng cách Euclid => bắt buộc scale trước.
    Xtr_s, Xte_s = _impute_scale(Xtr, Xte)
    a, b, c, d = relieff_extract_features(
        X_train=Xtr_s, y_train=ytr, X_test=Xte_s, y_test=yte,
        feature_names=feature_names, k=K_RELIEFF, out_dir=OUT_DIR, fold=fold, seed=SEED,
        n_iter=RELIEFF_N_ITER,
    )
    return a.astype(np.float32), b, c.astype(np.float32), d


def prep_mrmr(Xtr, ytr, Xte, yte, feature_names, fold):
    Xtr_s, Xte_s = _impute_scale(Xtr, Xte)
    a, b, c, d = mrmr_extract_features(
        X_train=Xtr_s, y_train=ytr, X_test=Xte_s, y_test=yte,
        feature_names=feature_names, k=K_MRMR, out_dir=OUT_DIR, fold=fold, seed=SEED,
        redundancy_weight=MRMR_REDUNDANCY_WEIGHT,
    )
    return a.astype(np.float32), b, c.astype(np.float32), d


def prep_elmo(Xtr, ytr, Xte, yte, feature_names, fold):
    """Pipeline ELMO (4 bước, leakage-free) trên train fold:
    impute -> scale -> L1 -> RFE -> IFS -> SBS. Trả về đặc trưng đã chọn,
    để mạng MLP DÙNG CHUNG huấn luyện như mọi method khác.
    """
    # Impute + scale bằng thống kê TRAIN (giống ELMO), không rò rỉ test.
    Xtr_i, Xte_i = ELMO.impute_mean(Xtr.copy(), Xte.copy())
    Xtr_s, Xte_s = ELMO.scale(Xtr_i, Xte_i)

    # Step 1 – L1
    mask, sel_features = ELMO.step1_l1_selection(Xtr_s, ytr, feature_names)
    if mask.sum() == 0:
        raise RuntimeError("Không đặc trưng nào sống sót sau L1.")
    Xtr_sel, Xte_sel = Xtr_s[:, mask], Xte_s[:, mask]

    # Step 2 – RFE ranking
    order = ELMO.step2_rfe_order(Xtr_sel, ytr)
    ranked_features = [sel_features[i] for i in order]
    Xtr_r, Xte_r = Xtr_sel[:, order], Xte_sel[:, order]

    # Step 3 – IFS (inner CV trên train): ELMO TỰ chọn số đặc trưng tối ưu
    best_k, ifs_features = ELMO.step3_ifs(Xtr_r, ytr, ranked_features)
    Xtr_ifs, Xte_ifs = Xtr_r[:, :best_k], Xte_r[:, :best_k]

    # Step 4 – SBS (inner CV trên train): tinh lọc thêm
    kept_cols, _ = ELMO.step4_sbs(Xtr_ifs, ytr, ifs_features)
    Xtr_f, Xte_f = Xtr_ifs[:, kept_cols], Xte_ifs[:, kept_cols]
    print(f"  ELMO selection      : L1 → RFE → IFS → SBS → {Xtr_f.shape[1]} features (adaptive)")

    return Xtr_f.astype(np.float32), ytr, Xte_f.astype(np.float32), yte


# Đăng ký 6 method (5 loại preprocessing gốc + ELMO 4 bước)
METHODS = {
    "FISHER":  prep_fisher,
    "L1":      prep_l1,
    "MRMR":    prep_mrmr,
    "RELIEFF": prep_relieff,
    "SNR":     prep_snr,
    "ELMO":    prep_elmo,
}


# ==============================================================================
# CHECKPOINT theo tung (method x fold): luu sau moi buoc -> resume khi chay lai
# ==============================================================================
import json as _json, os as _os

def _ckpt_path():
    return f"{OUT_DIR}/checkpoint_per_fold.json"

def _isnan(v):
    return v is None or (isinstance(v, float) and np.isnan(v))

def _rec_to_json(rec):
    out = {}
    for k, v in rec.items():
        if isinstance(v, np.integer):  v = int(v)
        if isinstance(v, np.floating): v = float(v)
        if isinstance(v, float) and np.isnan(v): v = None
        out[k] = v
    return out

def save_checkpoint(per_fold):
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    data = {name: [_rec_to_json(r) for r in recs] for name, recs in per_fold.items()}
    tmp = _ckpt_path() + ".tmp"
    with open(tmp, "w") as f:
        _json.dump(data, f, ensure_ascii=False, indent=2)
    _os.replace(tmp, _ckpt_path())          # ghi nguyen tu, tranh hong file giua chung

def load_checkpoint():
    p = _ckpt_path()
    if not Path(p).exists():
        return None
    try:
        with open(p) as f:
            data = _json.load(f)
    except Exception:
        return None
    for recs in data.values():
        for r in recs:
            for k, v in list(r.items()):
                if v is None and k != "Method":
                    r[k] = np.nan
    return data

def reset_checkpoint():
    """Goi tay (reset_checkpoint()) neu muon chay lai HOAN TOAN tu dau."""
    p = Path(_ckpt_path())
    if p.exists():
        p.unlink()
    print("Da xoa checkpoint ->", p)


# ==============================================================================
# BENCHMARK — Stratified K-Fold cross-validation
# ==============================================================================
def benchmark(X, y, feature_names, num_classes,
              n_folds=N_FOLDS, seed=SEED) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Chạy mỗi method qua Stratified K-Fold trên CÙNG các fold & CÙNG mạng MLP.

    Mỗi fold: chọn đặc trưng CHỈ trên train-fold (không rò rỉ test), train MLP
    dùng chung, đánh giá trên test-fold. Trả về:
      - summary    : bảng mean & std của 5 chỉ số (macro) qua các fold
      - fold_detail: bảng chi tiết từng method × từng fold
    """
    set_seed(seed)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    per_fold = load_checkpoint() or {name: [] for name in METHODS}
    for name in METHODS:
        per_fold.setdefault(name, [])
    _done = sum(len(v) for v in per_fold.values())
    if _done:
        print(f"[CHECKPOINT] resume: da co {_done} ket qua (method x fold), se bo qua phan da chay.")

    print(f"\n[CV] Stratified {n_folds}-Fold | total samples: {len(y)} | "
          f"classes: {num_classes}")

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        X_train_raw, X_test_raw = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]
        print("\n" + "#" * 70)
        print(f"FOLD {fold}/{n_folds} | Train: {len(y_train)} | Test: {len(y_test)}")
        print("#" * 70)

        for name, prep in METHODS.items():
            _done_fold = [r for r in per_fold[name] if int(r.get("Fold", -1)) == fold]
            if any(not _isnan(r.get("AUC")) for r in _done_fold):
                print(f"\n[Fold {fold}] METHOD: {name} -> da co trong checkpoint, bo qua")
                continue
            if _done_fold:                       # lan truoc chi co ban ghi loi -> chay lai
                per_fold[name] = [r for r in per_fold[name] if int(r.get("Fold", -1)) != fold]
            print("\n" + "=" * 70)
            print(f"[Fold {fold}] METHOD: {name}")
            print("=" * 70)
            try:
                Xtr_opt, ytr, Xte_opt, yte = prep(
                    X_train_raw, y_train, X_test_raw, y_test, feature_names, fold,
                )
                # ── Mạng MLP DÙNG CHUNG ─────────────────────────────────────
                model = step5_train_mlp(
                    X_train_opt=Xtr_opt, y_train=ytr, num_classes=num_classes,
                    seed=seed, **MLP_KWARGS,
                )
                m = evaluate_mlp(model, Xte_opt, yte)
                rec = {
                    "Fold":      fold,
                    "n_feat":    Xtr_opt.shape[1],
                    "AUC":       m["auc_macro"],
                    "Accuracy":  m["accuracy"],
                    "F1":        m["f1_macro"],
                    "Precision": m["precision_macro"],
                    "Recall":    m["recall_macro"],
                }
                print(f"  feat={rec['n_feat']:<5} | AUC={rec['AUC']:.4f} | "
                      f"ACC={rec['Accuracy']:.4f} | F1={rec['F1']:.4f} | "
                      f"P={rec['Precision']:.4f} | R={rec['Recall']:.4f}")
            except Exception as e:  # 1 method lỗi không làm chết cả benchmark
                print(f"  [ERROR] {name} thất bại ở fold {fold}: {e}")
                rec = {"Fold": fold, "n_feat": np.nan, "AUC": np.nan,
                       "Accuracy": np.nan, "F1": np.nan,
                       "Precision": np.nan, "Recall": np.nan}
            per_fold[name].append(rec)
            save_checkpoint(per_fold)          # luu ngay sau moi method

    # ── Tổng hợp mean / std qua các fold ──────────────────────────────────────
    metrics = ["AUC", "Accuracy", "F1", "Precision", "Recall"]
    summary_rows, fold_rows = [], []
    for name in METHODS:
        df_m = pd.DataFrame(per_fold[name])
        for _, r in df_m.iterrows():
            fold_rows.append({"Method": name, **r.to_dict()})
        row = {"Method": name, "n_feat": np.nanmean(df_m["n_feat"])}
        for met in metrics:
            row[f"{met}_mean"] = np.nanmean(df_m[met])
            row[f"{met}_std"]  = np.nanstd(df_m[met])
        summary_rows.append(row)

    summary     = pd.DataFrame(summary_rows).set_index("Method")
    fold_detail = pd.DataFrame(fold_rows)

    # Bảng "mean ± std" gọn để đọc
    pretty = pd.DataFrame(index=summary.index)
    pretty["n_feat"] = summary["n_feat"].round(1)
    for met in metrics:
        pretty[met] = (summary[f"{met}_mean"].map("{:.4f}".format)
                       + " ± " + summary[f"{met}_std"].map("{:.4f}".format))

    print("\n" + "=" * 96)
    print(f"BENCHMARK SUMMARY — Stratified {n_folds}-Fold (mean ± std, macro)")
    print("=" * 96)
    print(pretty.to_string())
    print("=" * 96)
    return summary, fold_detail


def main():
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    X, y, feature_names, num_classes = load_dataset()
    summary, fold_detail = benchmark(X, y, feature_names, num_classes)
    summary.to_csv(f"{OUT_DIR}/benchmark_results_summary.csv")
    fold_detail.to_csv(f"{OUT_DIR}/benchmark_results_per_fold.csv", index=False)
    print(f"\nĐã lưu kết quả (mean/std) -> {OUT_DIR}/benchmark_results_summary.csv")
    print(f"Đã lưu chi tiết từng fold -> {OUT_DIR}/benchmark_results_per_fold.csv")


## PAM50 — baseline feature selection theo tri thức miền
Khớp 50 gene PAM50 với feature của dataset, dùng các gene trùng để train MLP dùng chung.

In [ ]:
# ============================================================================
# PAM50 — feature selection theo tri thức miền (baseline phân loại subtype BRCA)
# Khớp 50 gene PAM50 với tên feature (dạng "omic__gene"), dùng các gene TRÙNG
# để train bằng mạng MLP DÙNG CHUNG, trả về đúng 5 metrics như các method khác.
# >>> Chạy cell này SAU cell benchmark (đã định nghĩa METHODS) và TRƯỚC cell main().
# ============================================================================
PAM50_GENES = [
    "ACTR3B", "ANLN", "BAG1", "BCL2", "BIRC5", "BLVRA", "CCNB1", "CCNE1", "CDC20", "CDC6",
    "CDH3", "CENPF", "CXXC5", "EGFR", "ERBB2", "ESR1", "EXO1", "FGFR4", "FOXA1", "FOXC1",
    "GPR160", "GRB7", "KRT14", "KRT17", "KRT5", "MAPT", "MDM2", "MELK", "MIA", "MKI67",
    "MLPH", "MMP11", "MYBL2", "MYC", "NAT1", "NOB1", "NUF2", "ORC6L", "PGR", "PHGDH",
    "PTTG1", "RRM2", "SFRP1", "SLC39A6", "TMEM45B", "TOP2A", "UBE2C", "UBE2T", "UHRF1", "WRN",
]

# Một số gene PAM50 có ký hiệu cũ/mới khác nhau giữa các bản TCGA -> map cho khỏi sót.
# (coi key/value là tương đương; bổ sung thêm nếu dataset của bạn dùng symbol khác.)
PAM50_ALIASES = {"ORC6L": "ORC6", "NUF2": "CDCA1"}

# Khớp với omics nào? PAM50 là chữ ký BIỂU HIỆN -> mặc định chỉ 'mRNA'.
# Đặt None để khớp trên MỌI omics (CNV/Methy/miRNA/mRNA).
PAM50_OMICS = ("mRNA",)


def _pam50_gene_token(fname):
    """'mRNA__ESR1' -> ('mRNA','ESR1');  'mRNA__ESR1|2099' -> ('mRNA','ESR1')."""
    omic, _, feat = fname.partition("__")
    gene = feat.split("|")[0].strip().upper()
    return omic, gene


def _pam50_aliases_of(gene_upper):
    """Tập ký hiệu tương đương (gồm chính nó) của 1 gene."""
    cands = {gene_upper}
    for k, v in PAM50_ALIASES.items():
        if gene_upper == k.upper():
            cands.add(v.upper())
        if gene_upper == v.upper():
            cands.add(k.upper())
    return cands


def pam50_coverage(feature_names, omics):
    """Tìm gene PAM50 trùng trong feature_names.
    Trả về: selected_idx, found_genes (canonical), missing_genes, per_omic_count.
    """
    present = {}                                  # gene_token -> [(idx, omic), ...]
    for i, f in enumerate(feature_names):
        omic, gene = _pam50_gene_token(f)
        if omics is not None and omic not in omics:
            continue
        present.setdefault(gene, []).append((i, omic))

    selected_idx, found, missing, per_omic = [], [], [], {}
    for g in PAM50_GENES:
        hit = False
        for c in _pam50_aliases_of(g.upper()):
            for (i, omic) in present.get(c, []):
                selected_idx.append(i)
                per_omic[omic] = per_omic.get(omic, 0) + 1
                hit = True
        (found if hit else missing).append(g)
    return selected_idx, found, missing, per_omic


def prep_pam50(Xtr, ytr, Xte, yte, feature_names, fold):
    idx, found, missing, per_omic = pam50_coverage(feature_names, PAM50_OMICS)

    # Nếu mRNA không khớp gene nào (vd cột là Ensembl ID) -> thử mọi omics.
    if len(idx) == 0 and PAM50_OMICS is not None:
        idx, found, missing, per_omic = pam50_coverage(feature_names, None)
        if idx:
            print("  [PAM50] mRNA không khớp gene nào -> fallback khớp trên MỌI omics")
    if len(idx) == 0:
        raise RuntimeError(
            "PAM50: không khớp được gene nào. Tên feature có thể không phải gene "
            "symbol (vd Ensembl ID) hoặc prefix omic khác — kiểm tra feature_names."
        )

    omic_str = ", ".join(f"{k}:{v}" for k, v in sorted(per_omic.items()))
    print(f"  PAM50 selection     : khớp {len(found)}/50 gene -> {len(idx)} feature ({omic_str})")
    if missing and fold == 1:
        print(f"  PAM50 thiếu {len(missing)} gene: {', '.join(missing)}")

    # Lưu bảng gene trùng/thiếu (1 lần, fold 1) để đưa vào báo cáo.
    if fold == 1:
        import pandas as _pd
        _pd.DataFrame({"pam50_gene": PAM50_GENES,
                       "matched": [g in set(found) for g in PAM50_GENES]}
                      ).to_csv(f"{OUT_DIR}/pam50_gene_overlap.csv", index=False)

    # Impute + scale bằng thống kê TRAIN (leakage-free), giống các method khác.
    Xtr_s, Xte_s = _impute_scale(Xtr[:, idx], Xte[:, idx])
    return Xtr_s.astype(np.float32), ytr, Xte_s.astype(np.float32), yte


# Đăng ký vào benchmark: chạy CHUNG mạng MLP, trả về 5 metrics như mọi method.
METHODS["PAM50"] = prep_pam50
print("Đã đăng ký method 'PAM50'. METHODS:", list(METHODS))


## 5. Chạy benchmark

In [12]:
main()


Loading omics data ...
  CNV     : 671 samples, 11203 features
  Methy   : 671 samples, 11189 features
  miRNA   : 671 samples, 310 features
  mRNA    : 671 samples, 11343 features

Merged matrix : 671 samples x 34045 features

[CV] Stratified 5-Fold | total samples: 671 | classes: 5

######################################################################
FOLD 1/5 | Train: 536 | Test: 135
######################################################################

[Fold 1] METHOD: FISHER
  feat=500   | AUC=0.9790 | ACC=0.8519 | F1=0.8482 | P=0.8107 | R=0.9161

[Fold 1] METHOD: L1
  feat=500   | AUC=0.9651 | ACC=0.8593 | F1=0.8367 | P=0.8601 | R=0.8257

[Fold 1] METHOD: MRMR
  mRMR selection      : 34,045 → 500 features
  Selected features saved → benchmark_out/mrmr_selected_features_fold1.csv
  feat=500   | AUC=0.8746 | ACC=0.6667 | F1=0.6302 | P=0.6017 | R=0.7574

[Fold 1] METHOD: RELIEFF
  ReliefF selection   : 34,045 → 500 features
  Selected features saved → benchmark_out/relieff_selecte